# Comerica Park Game & Weather Dataset (2015-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Comerica Park. It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Comerica Park'
HOME_TEAM = 'DET'
SEASONS = range(2015, 2026)  # 2015 through 2025
TIMEZONE = 'America/Detroit'

# Coordinates
STADIUM_LAT = 42.339474
STADIUM_LON = -83.049027

# Outfield directions (degrees from north)
CF_DIR = 157.5   # Center field: SSE
LCF_DIR = 137.5  # Left-center field: SE
RCF_DIR = 177.5  # Right-center field: S

# Output file
OUTPUT_FILE = 'tigers_data_2015.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Comerica Park
Home team: DET
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Timezone: America/Detroit


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Comerica Park home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Comerica Park games: 836
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2015    81
2016    79
2017    81
2018    81
2019    81
2020    28
2021    81
2022    81
2023    81
2024    81
2025    81
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   413656 2015-04-06    2015       MIN                 4                 0           4              2          12      2    15            229               92.3          3     51       0.0588   
1   413678 2015-04-08    2015       MIN                11                 0          11              0          11      9    19            315               84.5          1     59       0.0169   
2   413689 2015-04-09    2015       MIN                 7                 1           8     

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}\u00b0C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}\u00b0")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 26.5°C
Sample wind: 11.9 km/h from 209°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2015...
  2015: 6600 hourly records
Pulling weather for 2016...
  2016: 6600 hourly records
Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 836
Missing temp data: 0
      temp_c  rhum        pres  prcp       wspd        wdir  game_pk
0  12.866667  60.0  995.400000   0.0  10.400000  193.321930   413656
1   7.700000  86.0  993.100000   0.0   7.700000   11.236890   413678
2  13.633333  98.0  985.833333   4.5  21.466667  175.668332   413689


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Comerica Park outfield directions (degrees from north):
- Center field: ~157.5\u00b0 (SSE)
- Left-center field: ~137.5\u00b0 (SE)
- Right-center field: ~177.5\u00b0 (S)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  836.000000  836.000000  836.000000
mean    -0.270118   -0.093870   -0.413786
std     10.168272   10.051370   10.860357
min    -27.169710  -25.997225  -33.850247
25%     -7.467457   -7.258558   -8.019297
50%     -1.303017   -1.604071   -0.476335
75%      6.639558    6.686428    6.686331
max     33.716590   39.002869   37.893950


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

tigers_data = games_full[final_columns].copy()
tigers_data = tigers_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    tigers_data[col] = tigers_data[col].round(decimals)

print(f"Final dataset: {tigers_data.shape[0]} rows x {tigers_data.shape[1]} columns")

Final dataset: 836 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = tigers_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games (wider range for doubleheaders)
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(tigers_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = tigers_data[col].isna().sum()
    pct = 100 * n_null / len(tigers_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', tigers_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', tigers_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', tigers_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', tigers_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', tigers_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', tigers_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', tigers_data['temp_f'].mean(), '~60-75 F'),
    ('Min game temp', tigers_data['temp_f'].min(), '>25 F'),
    ('Max game temp', tigers_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', tigers_data['wspd'].mean(), '~8-18 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(tigers_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2015: 81 games [OK] (expected 75-100)
  2016: 79 games [OK] (expected 75-100)
  2017: 81 games [OK] (expected 75-100)
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 28 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  2025: 81 games [OK] (expected 75-100)
  TOTAL: 836 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 0 nulls (0.0%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 9.00 (expected ~8-10)
  Avg HR/ga

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
tigers_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,413656,2015-04-06,2015,MIN,2015-04-06 13:08:00,13,4,0,4,2,12,2,15,229,92.3,3,51,0.0588,0.1333,55.2,12.9,60.0,995.4,0.0,10.4,6.5,193.3,S,-8.43,-5.84,-10.01
1,413678,2015-04-08,2015,MIN,2015-04-08 13:08:00,13,11,0,11,0,11,9,19,315,84.5,1,59,0.0169,0.0000,45.9,7.7,86.0,993.1,0.0,7.7,4.8,11.2,N,6.40,4.55,7.48
2,413689,2015-04-09,2015,MIN,2015-04-09 13:08:00,13,7,1,8,1,8,7,15,243,87.4,4,56,0.0714,0.0667,56.5,13.6,98.0,985.8,4.5,21.5,13.3,175.7,S,-20.40,-16.88,-21.46
3,413793,2015-04-17,2015,CWS,2015-04-17 13:08:00,13,2,1,3,2,16,2,14,231,88.2,2,49,0.0408,0.1429,65.9,18.8,53.7,995.2,0.0,9.8,6.1,188.0,S,-8.44,-6.23,-9.64
4,413808,2015-04-18,2015,CWS,2015-04-18 13:08:00,13,3,12,15,3,13,7,24,318,85.3,3,64,0.0469,0.1250,61.7,16.5,43.0,997.0,0.0,25.1,15.6,44.3,NE,9.86,1.38,17.15
5,413823,2015-04-19,2015,CWS,2015-04-19 13:08:00,13,9,1,10,2,10,6,18,275,86.0,4,54,0.0741,0.1111,55.7,13.2,58.7,988.0,0.0,28.3,17.6,81.0,E,-6.61,-15.64,3.21
6,413840,2015-04-20,2015,NYY,2015-04-20 19:08:00,19,2,1,3,1,12,3,14,222,88.4,0,45,0.0000,0.0714,48.0,8.9,66.0,978.1,0.0,27.2,16.9,255.3,W,3.71,12.70,-5.73
7,413853,2015-04-21,2015,NYY,2015-04-21 19:08:00,19,2,5,7,2,14,12,18,313,88.1,1,53,0.0189,0.1111,47.7,8.7,43.3,981.0,0.0,21.3,13.2,253.3,W,2.16,9.28,-5.21
8,413868,2015-04-22,2015,NYY,2015-04-22 19:08:00,19,4,13,17,1,15,11,21,367,89.0,1,59,0.0169,0.0476,39.0,3.9,46.0,987.1,0.0,24.0,14.9,278.7,W,12.44,18.72,4.65
9,413881,2015-04-23,2015,NYY,2015-04-23 13:08:00,13,1,2,3,0,21,8,6,276,85.8,1,39,0.0256,0.0000,39.6,4.2,44.7,992.4,0.0,21.5,13.4,299.7,NW,17.01,20.50,11.46


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
tigers_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(tigers_data)}, Columns: {len(tigers_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == tigers_data.shape, f"Shape mismatch: {verify.shape} vs {tigers_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/tigers_data_2015.csv
File size: 125.1 KB
Rows: 836, Columns: 31

Save & reload verification: PASSED
